### Выбор сегмента: распределение матчей по режимам и типам лобби

Смотрим, какие сегменты есть в `RAW` после исключения анонимов (`account_id` = 0 / 4294967295) и матчей с нулевой длительностью. Цель — выбрать однородный сегмент для построения профилей игроков.

**Выводы:**
* **Ranked All Draft** (`game_mode=22`, `lobby_type=7`) — крупнейший сегмент: 172 905 матчей, 809 348 игроков, 1 403 550 строк «игрок × матч», средняя длительность 40.62 мин.
* **Turbo** (`game_mode=23`) — 98 829 матчей при средней длительности 25.25 мин. Темп игры принципиально другой, GPM/XPM несопоставимы с Ranked → исключён.
* **All Draft в обычном лобби** (`lobby_type=0`) — 25 797 матчей. Отличается от Ranked мотивацией игроков → исключён.
* Итоговый сегмент для дальнейшего анализа: `game_mode = 22 AND lobby_type = 7`.

In [3]:
from lakehouse_spark import SparkSessionContext

with SparkSessionContext('check-ml-player-features') as spark:
    spark.sql("""
SELECT
    m.game_mode,
    gm.name AS game_mode_name,
    m.lobby_type,
    lt.name AS lobby_type_name,
    COUNT(DISTINCT m.match_id) AS matches_count,
    COUNT(DISTINCT p.account_id) AS players_count,
    COUNT(*) AS player_match_rows,
    ROUND(AVG(m.duration) / 60.0, 2) AS avg_duration_min
FROM lakehouse.raw.pg__matches m
JOIN lakehouse.raw.pg__match_players p
    ON m.match_id = p.match_id
LEFT JOIN lakehouse.raw.pg__game_modes gm
    ON m.game_mode = gm.mode_id
LEFT JOIN lakehouse.raw.pg__lobby_types lt
    ON m.lobby_type = lt.lobby_id
WHERE p.account_id IS NOT NULL
  AND p.account_id NOT IN (0, 4294967295)
  AND m.duration > 0
  AND m.game_mode IN (22, 23)
  AND m.lobby_type IN (0, 7)
GROUP BY
    m.game_mode,
    gm.name,
    m.lobby_type,
    lt.name
ORDER BY matches_count DESC;
    """).show(truncate=False)

+---------+-------------------+----------+-----------------+-------------+-------------+-----------------+----------------+
|game_mode|game_mode_name     |lobby_type|lobby_type_name  |matches_count|players_count|player_match_rows|avg_duration_min|
+---------+-------------------+----------+-----------------+-------------+-------------+-----------------+----------------+
|22       |game_mode_all_draft|7         |lobby_type_ranked|172905       |809348       |1403550          |40.62           |
|23       |game_mode_turbo    |0         |lobby_type_normal|98829        |457985       |838745           |25.25           |
|22       |game_mode_all_draft|0         |lobby_type_normal|25797        |131877       |207430           |39.75           |
+---------+-------------------+----------+-----------------+-------------+-------------+-----------------+----------------+



### Профиль игрока на Ranked All Draft: агрегация и проверки качества

Строим первую версию витрины признаков: одна строка = один `account_id`, сырые средние по всем матчам аккаунта. Проверяем схему, пропуски, уникальность ключа и диапазоны значений.

**Выводы:**
* Получено **143 208 профилей × 17 признаков**. Пропусков нет ни в одной колонке, дубликатов `account_id` нет.
* Диапазон `matches_count`: 3–26. Порог `HAVING COUNT(DISTINCT match_id) >= 3` применён на этапе агрегации.
* `short_match_share = 0.000` у всех профилей — матчей короче 7 минут в сегменте нет. Признак неинформативен, подлежит удалению.
* `win_rate` принимает только значения 0 / 33.33 / 66.67 / 100 при медиане 3 матча. Сырой винрейт дискретен и ненадёжен → требуется сглаживание.
* Экстремумы эффективности: `avg_gpm` до 1198.4, `avg_xpm` до 1642.0, `avg_kda` до 45.33. Это потенциальные аномалии, но без нормировки на героя и роль их нельзя интерпретировать.
* Смешанные типы данных: часть колонок имеет тип `decimal` вместо `double` (`avg_hero_damage_per_min`, `avg_tower_damage_per_min`, `win_rate`, `median_duration_min`, `short_match_share`). Потребуется приведение к `double` перед подачей в модель.

In [4]:
from lakehouse_spark import SparkSessionContext

# ============================================================
# 1. Подключение к Spark
# ============================================================

with SparkSessionContext('ml-player-features-ranked') as spark:

    # ========================================================
    # 2. Получение данных и построение признаков
    # ========================================================

    ml_player_features_ranked = spark.sql("""

        SELECT
            -- Идентификатор игрока
            p.account_id,

            -- 1. Активность игрока
            COUNT(DISTINCT p.match_id) AS matches_count,

            -- 2. Эффективность игрока
            ROUND(AVG(p.kda), 2) AS avg_kda,
            ROUND(AVG(p.gold_per_min), 2) AS avg_gpm,
            ROUND(AVG(p.xp_per_min), 2) AS avg_xpm,

            ROUND(AVG(p.kills), 2) AS avg_kills,
            ROUND(AVG(p.deaths), 2) AS avg_deaths,
            ROUND(AVG(p.assists), 2) AS avg_assists,
            ROUND(AVG(p.level), 2) AS avg_level,

            -- 3. Урон
            ROUND(AVG(p.hero_damage), 2) AS avg_hero_damage,
            ROUND(AVG(p.tower_damage), 2) AS avg_tower_damage,

            -- Урон с учётом продолжительности матча
            ROUND(
                AVG(p.hero_damage / (m.duration / 60.0)),
                2
            ) AS avg_hero_damage_per_min,

            ROUND(
                AVG(p.tower_damage / (m.duration / 60.0)),
                2
            ) AS avg_tower_damage_per_min,

            -- 4. Результативность
            ROUND(
                100.0 * AVG(
                    CASE
                        WHEN p.win = true THEN 1.0
                        ELSE 0.0
                    END
                ),
                2
            ) AS win_rate,

            -- 5. Продолжительность матчей
            ROUND(
                AVG(m.duration) / 60.0,
                2
            ) AS avg_duration_min,

            ROUND(
                percentile_approx(m.duration, 0.5) / 60.0,
                2
            ) AS median_duration_min,

            -- Доля матчей короче 7 минут
            ROUND(
                AVG(
                    CASE
                        WHEN m.duration < 420 THEN 1.0
                        ELSE 0.0
                    END
                ),
                3
            ) AS short_match_share

        FROM lakehouse.raw.pg__match_players p

        INNER JOIN lakehouse.raw.pg__matches m
            ON p.match_id = m.match_id

        WHERE
            -- Исключаем неизвестные идентификаторы игроков
            p.account_id IS NOT NULL
            AND p.account_id NOT IN (0, 4294967295)

            -- Оставляем только All Draft Ranked
            AND m.game_mode = 22
            AND m.lobby_type = 7

            -- Исключаем матчи с некорректной длительностью
            AND m.duration > 0

        GROUP BY
            p.account_id

        -- Минимум три уникальных матча на игрока
        HAVING COUNT(DISTINCT p.match_id) >= 3

    """)

    # ========================================================
    # 3. Получение количества профилей
    # ========================================================

    print("Размер подготовленного набора данных:")

    print(
        "Количество игроков:",
        ml_player_features_ranked.count()
    )

    print(
        "Количество признаков:",
        len(ml_player_features_ranked.columns)
    )

    # ========================================================
    # 4. Проверка структуры
    # ========================================================

    print("\nСтруктура DataFrame:")

    ml_player_features_ranked.printSchema()

    # ========================================================
    # 5. Проверка первых строк
    # ========================================================

    print("\nПервые 10 профилей игроков:")

    ml_player_features_ranked.show(
        10,
        truncate=False
    )

    # ========================================================
    # 6. Проверка пропусков
    # ========================================================

    from pyspark.sql import functions as F

    null_checks = ml_player_features_ranked.select(
        [
            F.count(
                F.when(F.col(column).isNull(), 1)
            ).alias(column)
            for column in ml_player_features_ranked.columns
        ]
    )

    print("\nКоличество пропусков по столбцам:")

    null_checks.show(truncate=False)

    # ========================================================
    # 7. Проверка уникальности игроков
    # ========================================================

    total_rows = ml_player_features_ranked.count()

    unique_players = (
        ml_player_features_ranked
        .select("account_id")
        .distinct()
        .count()
    )

    print("\nПроверка уникальности:")

    print("Всего строк:", total_rows)
    print("Уникальных игроков:", unique_players)
    print("Есть дубликаты:", total_rows != unique_players)

    # ========================================================
    # 8. Проверка диапазонов показателей
    # ========================================================

    print("\nМинимальные и максимальные значения:")

    ml_player_features_ranked.select(
        F.min("matches_count").alias("min_matches"),
        F.max("matches_count").alias("max_matches"),

        F.min("win_rate").alias("min_win_rate"),
        F.max("win_rate").alias("max_win_rate"),

        F.min("short_match_share").alias("min_short_share"),
        F.max("short_match_share").alias("max_short_share"),

        F.min("avg_gpm").alias("min_gpm"),
        F.max("avg_gpm").alias("max_gpm"),

        F.min("avg_xpm").alias("min_xpm"),
        F.max("avg_xpm").alias("max_xpm")
    ).show(truncate=False)

    # ========================================================
    # 9. Перенос в Pandas для последующего анализа
    # ========================================================

    ml_player_features_ranked_pd = (
        ml_player_features_ranked.toPandas()
    )

    print("\nDataFrame для Python / Pandas:")

    print(
        "Размер:",
        ml_player_features_ranked_pd.shape
    )

    display(
        ml_player_features_ranked_pd.head()
    )

Размер подготовленного набора данных:


Количество игроков: 143208
Количество признаков: 17

Структура DataFrame:
root
 |-- account_id: long (nullable = true)
 |-- matches_count: long (nullable = false)
 |-- avg_kda: double (nullable = true)
 |-- avg_gpm: double (nullable = true)
 |-- avg_xpm: double (nullable = true)
 |-- avg_kills: double (nullable = true)
 |-- avg_deaths: double (nullable = true)
 |-- avg_assists: double (nullable = true)
 |-- avg_level: double (nullable = true)
 |-- avg_hero_damage: double (nullable = true)
 |-- avg_tower_damage: double (nullable = true)
 |-- avg_hero_damage_per_min: decimal(19,2) (nullable = true)
 |-- avg_tower_damage_per_min: decimal(19,2) (nullable = true)
 |-- win_rate: decimal(8,2) (nullable = true)
 |-- avg_duration_min: double (nullable = true)
 |-- median_duration_min: decimal(14,2) (nullable = true)
 |-- short_match_share: decimal(5,3) (nullable = true)


Первые 10 профилей игроков:


+----------+-------------+-------+-------+-------+---------+----------+-----------+---------+---------------+----------------+-----------------------+------------------------+--------+----------------+-------------------+-----------------+
|account_id|matches_count|avg_kda|avg_gpm|avg_xpm|avg_kills|avg_deaths|avg_assists|avg_level|avg_hero_damage|avg_tower_damage|avg_hero_damage_per_min|avg_tower_damage_per_min|win_rate|avg_duration_min|median_duration_min|short_match_share|
+----------+-------------+-------+-------+-------+---------+----------+-----------+---------+---------------+----------------+-----------------------+------------------------+--------+----------------+-------------------+-----------------+
|13000     |3            |4.03   |385.67 |596.67 |3.33     |8.33      |25.67      |21.33    |14797.67       |783.67          |340.82                 |18.04                   |100.00  |43.73           |42.73              |0.000            |
|32113     |3            |2.31   |461.33

+----------+-------------+-------+-------+-------+---------+----------+-----------+---------+---------------+----------------+-----------------------+------------------------+--------+----------------+-------------------+-----------------+
|account_id|matches_count|avg_kda|avg_gpm|avg_xpm|avg_kills|avg_deaths|avg_assists|avg_level|avg_hero_damage|avg_tower_damage|avg_hero_damage_per_min|avg_tower_damage_per_min|win_rate|avg_duration_min|median_duration_min|short_match_share|
+----------+-------------+-------+-------+-------+---------+----------+-----------+---------+---------------+----------------+-----------------------+------------------------+--------+----------------+-------------------+-----------------+
|0         |0            |0      |0      |0      |0        |0         |0          |0        |0              |0               |0                      |0                       |0       |0               |0                  |0                |
+----------+-------------+-------+------


Проверка уникальности:
Всего строк: 143208
Уникальных игроков: 143208
Есть дубликаты: False

Минимальные и максимальные значения:


+-----------+-----------+------------+------------+---------------+---------------+-------+-------+-------+-------+
|min_matches|max_matches|min_win_rate|max_win_rate|min_short_share|max_short_share|min_gpm|max_gpm|min_xpm|max_xpm|
+-----------+-----------+------------+------------+---------------+---------------+-------+-------+-------+-------+
|3          |26         |0.00        |100.00      |0.000          |0.000          |106.67 |1198.4 |67.0   |1642.0 |
+-----------+-----------+------------+------------+---------------+---------------+-------+-------+-------+-------+




DataFrame для Python / Pandas:
Размер: (143208, 17)


,account_id,matches_count,avg_kda,avg_gpm,avg_xpm,avg_kills,avg_deaths,avg_assists,avg_level,avg_hero_damage,avg_tower_damage,avg_hero_damage_per_min,avg_tower_damage_per_min,win_rate,avg_duration_min,median_duration_min,short_match_share
0,13000,3,4.03,385.67,596.67,3.33,8.33,25.67,21.33,14797.67,783.67,340.82,18.04,100.00,43.73,42.73,0.000
1,32113,3,2.31,461.33,619.00,6.67,8.33,11.67,20.67,25160.33,1299.00,600.43,30.72,0.00,40.83,43.18,0.000
2,34147,5,2.90,342.80,643.80,8.60,8.80,17.00,22.40,22407.00,607.20,477.53,12.88,60.00,46.27,46.37,0.000
3,44365,3,1.06,443.33,537.00,3.00,10.00,8.33,20.00,14190.33,4565.33,317.30,96.59,33.33,42.07,46.90,0.000
4,62015,3,4.09,329.00,575.33,7.00,8.33,23.33,20.67,15380.67,1223.67,361.95,26.68,66.67,42.56,44.43,0.000


### Описательная статистика признаков baseline-витрины

Считаем перцентили по числовым признакам, чтобы понять форму распределений и оценить выраженность правых хвостов.

**Выводы:**
* **Медиана `matches_count` = 3**, p75 = 4, p99 = 8. Большинство профилей построены на минимально допустимом числе матчей — это ключевое ограничение выборки.
* `avg_gpm`: медиана 476.25, p99 = 787.74, max 1198.4. Распределение близко к симметричному (mean 484 ≈ median 476), правый хвост умеренный.
* `avg_tower_damage`: медиана 2266.67 при mean 3304.05 и std 3245.52 — сильная правая асимметрия, стандартное отклонение сопоставимо со средним. Признак зависит от роли и длительности матча.
* `avg_kda`: медиана 3.44, p99 = 15.63, max 45.33 — тяжёлый правый хвост. При n = 3 такие значения могут быть случайными.
* `avg_duration_min`: медиана 39.89, диапазон 11.51–71.73. Разброс длительности подтверждает необходимость нормировки показателей на минуту или через внутриматчевой z-score.

In [5]:
# Столбцы с числовыми характеристиками игроков
feature_columns = [
    'matches_count',
    'avg_kda',
    'avg_gpm',
    'avg_xpm',
    'avg_kills',
    'avg_deaths',
    'avg_assists',
    'avg_level',
    'avg_hero_damage',
    'avg_tower_damage',
    'avg_hero_damage_per_min',
    'avg_tower_damage_per_min',
    'win_rate',
    'avg_duration_min',
    'median_duration_min'
]

# Описательная статистика
profile_stats = (
    ml_player_features_ranked_pd[feature_columns]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
    .T
)

display(profile_stats.round(2))

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
matches_count,143208.0,3.71,1.17,3.00,3.00,3.00,4.00,5.00,6.00,8.00,26.00
avg_kda,143208.0,4.34,3.03,0.00,2.41,3.44,5.19,8.08,10.52,15.63,45.33
avg_gpm,143208.0,484.28,120.96,106.67,390.33,476.25,566.25,647.00,697.00,787.74,1198.40
avg_xpm,143208.0,691.25,166.71,67.00,573.00,677.67,797.33,913.33,983.50,1124.33,1642.00
avg_kills,143208.0,8.09,3.79,0.00,5.33,7.50,10.33,13.20,15.00,19.25,42.00
avg_deaths,143208.0,8.13,2.70,0.00,6.25,8.00,10.00,11.67,12.75,15.00,22.67
avg_assists,143208.0,14.76,5.15,0.00,11.00,14.33,18.00,21.67,23.75,28.00,42.00
avg_level,143208.0,21.26,3.08,3.00,19.33,21.33,23.33,25.00,26.00,27.67,30.00
avg_hero_damage,143208.0,25372.52,9771.99,23.00,18387.23,24036.16,30900.25,38309.97,43317.30,54154.95,99633.33
avg_tower_damage,143208.0,3304.05,3245.52,0.00,879.00,2266.67,4727.23,7799.10,9968.44,14397.96,27752.00


### Влияние порога по числу матчей на размер выборки

Проверяем, сколько профилей останется при повышении минимального числа матчей. Нужно для оценки компромисса «надёжность статистики vs размер выборки».

**Выводы:**
* Порог **3 матча**: 143 208 игроков (100%).
* Порог **5 матчей**: 24 060 игроков (16.80%) — потеря 83% выборки.
* Порог **10 матчей**: 491 игрок (0.34%) — выборка непригодна для обучения модели.
* Порог **15+ матчей**: менее 100 игроков.
* **Итог:** повысить порог невозможно без потери выборки. Окно наблюдения (18 часов) физически не позволяет набрать больше матчей на аккаунт. Вместо повышения порога малое `n` компенсируется другими способами: сглаженный винрейт, признаки устойчивости (нижний квартиль), оценка доли шума через перестановочный тест.

In [6]:
# Сколько игроков остаётся при разных порогах матчей

thresholds = [3, 5, 10, 15, 20]

for threshold in thresholds:
    players_count = (
        ml_player_features_ranked_pd[
            ml_player_features_ranked_pd['matches_count'] >= threshold
        ].shape[0]
    )

    share = players_count / len(ml_player_features_ranked_pd) * 100

    print(
        f'Минимум {threshold:2} матчей: '
        f'{players_count:6} игроков ({share:.2f}%)'
    )

Минимум  3 матчей: 143208 игроков (100.00%)
Минимум  5 матчей:  24060 игроков (16.80%)
Минимум 10 матчей:    491 игроков (0.34%)
Минимум 15 матчей:     99 игроков (0.07%)
Минимум 20 матчей:     20 игроков (0.01%)


26/11/13 09:00:27 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-de90d372-6f58-4abb-b147-162e4eb7fb2d. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-de90d372-6f58-4abb-b147-162e4eb7fb2d
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:67)
	at scala.collection.ArrayOps$.foreach$exten